In [1]:
# 1. Réinitialiser et cloner proprement le dépôt GitHub
%cd /content
!rm -rf harmonization-framework
!git clone https://github.com/bmir-radx/harmonization-framework

# 2. Installer la librairie de manière globale pour Colab
%cd /content/harmonization-framework
%pip install -r requirements.txt
!pip install .

# 3. Forcer l'installation de la version compatible de pint sous Python 3.13
%pip install --quiet "pint==0.24.4"

# 4. Se déplacer dans le dossier demo pour accéder directement aux fichiers CSV de test
%cd /content/harmonization-framework/demo

/content
Cloning into 'harmonization-framework'...
remote: Enumerating objects: 1136, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 1136 (delta 27), reused 19 (delta 19), pack-reused 1097 (from 2)
Receiving objects: 100% (1136/1136), 308.83 KiB | 6.71 MiB/s, done.
Resolving deltas: 100% (632/632), done.
/content/harmonization-framework
  Using cached flexparser-0.3.1-py3-none-any.whl.metadata (18 kB)
  Using cached Pint-0.24.3-py3-none-any.whl.metadata (8.5 kB)
Using cached flexparser-0.3.1-py3-none-any.whl (27 kB)
Using cached Pint-0.24.3-py3-none-any.whl (301 kB)
  Attempting uninstall: flexparser
    Found existing installation: flexparser 0.4
    Uninstalling flexparser-0.4:
      Successfully uninstalled flexparser-0.4
  Attempting uninstall: pint
    Found existing installation: Pint 0.24.4
    Uninstalling Pint-0.24.4:
      Successfully uninstalled Pint-0.24.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [p

In [2]:
import sys
# Forcer le chemin local de la bibliothèque en première position
local_path = '/content/harmonization-framework/src'
if local_path not in sys.path:
    sys.path.insert(0, local_path)

# Supprimer les anciens modules en cache pour forcer le rechargement depuis le dossier local
for module in list(sys.modules.keys()):
    if module.startswith('harmonization_framework'):
        del sys.modules[module]

import pandas as pd
from harmonization_framework.harmonization_rule import HarmonizationRule
from harmonization_framework.primitives import Cast, Bin, ConvertUnits, EnumToEnum, Round
from harmonization_framework.rule_registry import RuleSet as RuleStore
from harmonization_framework import harmonize as transformations

### Step 1: Set up logger

In [3]:
import harmonization_framework.replay_log as rlog
logger = rlog.configure_logger(3, "demo.log")

In [6]:
import os
# S'assurer que le répertoire de travail est bien le dossier de démonstration contenant les CSV
demo_dir = "/content/harmonization-framework/demo"
if os.path.exists(demo_dir):
    os.chdir(demo_dir)

df_source1 = pd.read_csv(
    "demo_source1.csv",
    index_col=0,
    converters={
        "current_employment_status": int,
        "commute_distance_miles": float,
    },
)
df_source1 = df_source1.drop(["edu_years_of_school", "zip_code_9"], axis=1)

df_source2 = pd.read_csv(
    "demo_source2.csv",
    index_col=0,
    converters={
        "employment": int,
        "commute_distance_km": float,
    },
)
df_source2 = df_source2.drop(["edu_years_of_school", "zip_code_9"], axis=1)

In [14]:
print(df_source1)

    current_employment_status  commute_distance_miles
id                                                   
0                           7                    4.70
1                           2                    6.69
2                           1                    8.44
3                           3                    0.00
4                           1                    4.62


In [13]:
print(df_source2)

    employment  commute_distance_km
id                                 
0            1                17.73
1            2                 4.45
2            2                 9.12
3            3                 0.00
4            3                 0.00


### Step 3: Define harmonization rules to suit use case.

In [9]:
# spin up a data store for our harmonization rules
rules = RuleStore()

In [10]:
# re-coding from UP to Data Hub data element
employment_rule_up = HarmonizationRule(
    sources=["current_employment_status"],
    target="nih_employment",
    transformation=[
        EnumToEnum({
            1: 0,
            2: 97,
            3: 1,
            4: 4,
            5: 5,
            6: 97,
            7: 2,
            96: 97,
            98: 99,
            99: 98,
        }),
    ]
)
rules.add_rule(employment_rule_up)

# re-coding from rad and Data Hub data element
employment_rule_rad = HarmonizationRule(
    sources=["employment"],
    target="nih_employment",
    transformation=[
        EnumToEnum({
            1: 0,
            2: 0,
            3: 1,
        }),
    ]
)
rules.add_rule(employment_rule_rad)

In [11]:
# kilometer to miles conversion, mainting 2 significant digits
dist_rule_rad = HarmonizationRule(
    sources=["commute_distance_km"],
    target="commute_distance_miles",
    transformation=[
        ConvertUnits(source="kilometers", target="miles"),
        Round(precision=2),
    ]
)
rules.add_rule(dist_rule_rad)

### Step 4: Execute transformations and combine datasets

In [12]:
# Créer des registres de règles distincts pour éviter l'écrasement des clés cibles communes
rules_up = RuleStore()
rules_up.add_rule(employment_rule_up)

rules_rad = RuleStore()
rules_rad.add_rule(employment_rule_rad)
rules_rad.add_rule(dist_rule_rad)

# Exécuter l'harmonisation avec leurs registres respectifs
df_harmonized1 = transformations.harmonize_dataset(df_source1, rules_up, dataset_name="up_dataset", logger=logger)
df_harmonized2 = transformations.harmonize_dataset(df_source2, rules_rad, dataset_name="rad_dataset", logger=logger)

# Combiner directement les DataFrames à l'aide de pandas
integrated = pd.concat([df_harmonized1, df_harmonized2], ignore_index=True)
integrated

INFO:ReplayLogger:{"event": "rule", "action": {"sources": ["current_employment_status"], "target": "nih_employment", "operations": [{"operation": "enum_to_enum", "mapping": [{"from": 1, "to": 0}, {"from": 2, "to": 97}, {"from": 3, "to": 1}, {"from": 4, "to": 4}, {"from": 5, "to": 5}, {"from": 6, "to": 97}, {"from": 7, "to": 2}, {"from": 96, "to": 97}, {"from": 98, "to": 99}, {"from": 99, "to": 98}], "strict": false}]}, "dataset": "up_dataset"}
INFO:ReplayLogger:{"event": "rule", "action": {"sources": ["employment"], "target": "nih_employment", "operations": [{"operation": "enum_to_enum", "mapping": [{"from": 1, "to": 0}, {"from": 2, "to": 0}, {"from": 3, "to": 1}], "strict": false}]}, "dataset": "rad_dataset"}
INFO:ReplayLogger:{"event": "rule", "action": {"sources": ["commute_distance_km"], "target": "commute_distance_miles", "operations": [{"operation": "convert_units", "source_unit": "kilometers", "target_unit": "miles"}, {"operation": "round", "precision": 2}]}, "dataset": "rad_dat

Applying rule -> nih_employment (sources: ['current_employment_status'])
Applying rule -> nih_employment (sources: ['employment'])
Applying rule -> commute_distance_miles (sources: ['commute_distance_km'])


,current_employment_status,commute_distance_miles,nih_employment,source dataset,original_id,employment,commute_distance_km
0,7.0,4.70,2,up_dataset,0,NaN,NaN
1,2.0,6.69,97,up_dataset,1,NaN,NaN
2,1.0,8.44,0,up_dataset,2,NaN,NaN
3,3.0,0.00,1,up_dataset,3,NaN,NaN
4,1.0,4.62,0,up_dataset,4,NaN,NaN
5,NaN,11.02,0,rad_dataset,0,1.0,17.73
6,NaN,2.77,0,rad_dataset,1,2.0,4.45
7,NaN,5.67,0,rad_dataset,2,2.0,9.12
8,NaN,0.00,1,rad_dataset,3,3.0,0.00
9,NaN,0.00,1,rad_dataset,4,3.0,0.00


In [42]:
integrated

,current_employment_status,commute_distance_miles,nih_employment,source dataset,original_id,employment,commute_distance_km
0,7.0,4.70,2,up_dataset,0,NaN,NaN
1,2.0,6.69,97,up_dataset,1,NaN,NaN
2,1.0,8.44,0,up_dataset,2,NaN,NaN
3,3.0,0.00,1,up_dataset,3,NaN,NaN
4,1.0,4.62,0,up_dataset,4,NaN,NaN
5,NaN,11.02,0,rad_dataset,0,1.0,17.73
6,NaN,2.77,0,rad_dataset,1,2.0,4.45
7,NaN,5.67,0,rad_dataset,2,2.0,9.12
8,NaN,0.00,1,rad_dataset,3,3.0,0.00
9,NaN,0.00,1,rad_dataset,4,3.0,0.00


### Example: save harmonization rules to disk

In [15]:
rules.save("demo_rules.json")

### Example: replay harmonization from log

In [20]:
import harmonization_framework.replay_log.replay_logger as replay_logger
from harmonization_framework.replay_log.replay_logger import replay

# Rejouer l'harmonisation depuis le fichier de log
replay_results = replay("demo.log", {"up_dataset": df_source1, "rad_dataset": df_source2})

# Combiner les jeux de données rejoués à l'aide de pandas
import pandas as pd
replay_integrated = pd.concat(replay_results.values(), ignore_index=True)
replay_integrated

ImportError: cannot import name 'replay' from 'harmonization_framework.replay_log.replay_logger' (/content/harmonization-framework/src/harmonization_framework/replay_log/replay_logger.py)

In [ ]:
replay_integrated

,nih_employment,commute_distance_miles,source dataset,original_id
id,,,,
0,2,4.70,up_dataset,0
1,97,6.69,up_dataset,1
2,0,8.44,up_dataset,2
3,1,0.00,up_dataset,3
4,0,4.62,up_dataset,4
5,0,11.02,rad_dataset,0
6,0,2.77,rad_dataset,1
7,0,5.67,rad_dataset,2
8,1,0.00,rad_dataset,3


### Example: harmonization rule using categorical bins

In [ ]:
# age in text to age range
age_rule = HarmonizationRule(
    source="age_text",
    target="age_range",
    transformation=[
        Cast("text", "integer"),
        Bin([
            [0, [0, 30]],
            [1, [31, 40]],
            [2, [41, 50]],
            [3, [51, 60]],
            [4, [61, 70]],
            [5, [70, sys.maxsize]],
        ]),
    ],
)
rules.add_rule(age_rule)